# Two-Dimensional Adjoint Gradient Verification

Central finite differences are compared with adjoint directional derivatives
for relative permittivity and conductivity. Orders 2, 4, and 8 are tested with
float32 wavefield storage and no temporal subsampling.


In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
candidates = [cwd, cwd / "tests"]
candidates.extend(parent / "tests" for parent in cwd.parents)
NOTEBOOK_DIR = next(
    (path for path in candidates if (path / "verification_utils.py").is_file()),
    None,
)
if NOTEBOOK_DIR is None:
    raise FileNotFoundError("verification_utils.py was not found from the current directory.")
notebook_path = str(NOTEBOOK_DIR)
if notebook_path not in sys.path:
    sys.path.insert(0, notebook_path)

import verification_utils as vu

REPO_ROOT = vu.configure_local_import()
for module_name in tuple(sys.modules):
    if module_name == "DeepGPR" or module_name.startswith("DeepGPR."):
        del sys.modules[module_name]
import DeepGPR

LOADED_PACKAGE = vu.assert_local_deepgpr(DeepGPR, REPO_ROOT)
print(f"Repository root: {REPO_ROOT}")
print(f"DeepGPR package: {LOADED_PACKAGE}")


Repository root: /Users/llsra/Desktop/DeepGPR
DeepGPR package: /Users/llsra/Desktop/DeepGPR/src/DeepGPR/__init__.py


In [2]:
import torch

torch.manual_seed(2026)
DEVICE = torch.device("cpu")
CHECKS = []
METADATA = vu.runtime_metadata(DeepGPR, DEVICE)
nx, ny, nt = 24, 30, 180
dx, dt, pml = 0.02, 3.0e-11, 4
x = torch.arange(nx, dtype=torch.float32)[:, None]
y = torch.arange(ny, dtype=torch.float32)[None, :]
anomaly = torch.exp(
    -0.5 * (((x - 15.0) / 3.5) ** 2 + ((y - 17.0) / 4.5) ** 2)
)
er0 = torch.full((nx, ny), 4.0)
se0 = torch.full((nx, ny), 3.0e-4)
er_true = er0 + 0.35 * anomaly
se_true = se0 + 1.5e-4 * anomaly
source_location = torch.tensor([[[6, 10, 0]]], dtype=torch.int32)
receiver_location = torch.tensor(
    [[[6, 14, 0], [6, 18, 0], [6, 22, 0]]], dtype=torch.int32
)
source = DeepGPR.wavelet.ricker(2.5e8, nt, dt, 4.0e-9).reshape(1, nt, 1)
interior = vu.normalized_interior_mask((nx, ny), pml, DEVICE)
boundary = ~interior


In [3]:
all_rows = []
for order in (2, 4, 8):
    def simulate(er_value, se_value):
        return DeepGPR.compute(
            device=DEVICE,
            dx=dx,
            dt=dt,
            source_amplitudes=source,
            source_location=source_location,
            receiver_location=receiver_location,
            er=er_value,
            se=se_value,
            pmlthick=pml,
            fdtd_order=order,
            mode=2,
            model_gradient_sampling_interval=1,
            wavefield_storage_dtype=torch.float32,
        )[-1]

    with torch.no_grad():
        observed = simulate(er_true, se_true)
    data_scale = observed.abs().max().clamp_min(1.0e-12)

    def objective(er_value, se_value):
        residual = (simulate(er_value, se_value) - observed) / data_scale
        return 0.5 * residual.square().sum()

    er = er0.clone().requires_grad_(True)
    se = se0.clone().requires_grad_(True)
    loss = objective(er, se)
    loss.backward()
    vu.assert_finite(f"order {order} gradients", er.grad, se.grad)

    direction_er = vu.gradient_direction(er.grad, interior)
    direction_se = vu.gradient_direction(se.grad, interior)
    rows_er = vu.directional_derivative_rows(
        lambda value: objective(value, se.detach()),
        er.detach(),
        direction_er,
        er.grad,
        (8.0e-2, 4.0e-2, 2.0e-2, 1.0e-2),
    )
    rows_se = vu.directional_derivative_rows(
        lambda value: objective(er.detach(), value),
        se.detach(),
        direction_se,
        se.grad,
        (5.0e-4, 2.0e-4, 1.0e-4, 5.0e-5),
    )
    best_er = vu.best_relative_error(rows_er)
    best_se = vu.best_relative_error(rows_se)
    row = {
        "order": order,
        "loss": float(loss.detach()),
        "er_best_relative_error": best_er,
        "se_best_relative_error": best_se,
        "er_rows": rows_er,
        "se_rows": rows_se,
    }
    all_rows.append(row)
    vu.record_check(
        CHECKS,
        f"order {order} relative-permittivity directional derivative",
        best_er < 5.0e-3,
        best_relative_error=best_er,
        tolerance=5.0e-3,
    )
    vu.record_check(
        CHECKS,
        f"order {order} conductivity directional derivative",
        best_se < 5.0e-3,
        best_relative_error=best_se,
        tolerance=5.0e-3,
    )
    vu.record_check(
        CHECKS,
        f"order {order} CPML material-gradient exclusion",
        vu.boundary_absmax(er.grad, boundary) == 0.0
        and vu.boundary_absmax(se.grad, boundary) == 0.0,
        er_boundary_absmax=vu.boundary_absmax(er.grad, boundary),
        se_boundary_absmax=vu.boundary_absmax(se.grad, boundary),
    )


[PASS] order 2 relative-permittivity directional derivative
{
  "best_relative_error": 5.78340104015241e-05,
  "tolerance": 0.005
}
[PASS] order 2 conductivity directional derivative
{
  "best_relative_error": 1.593099810383998e-05,
  "tolerance": 0.005
}
[PASS] order 2 CPML material-gradient exclusion
{
  "er_boundary_absmax": 0.0,
  "se_boundary_absmax": 0.0
}
[PASS] order 4 relative-permittivity directional derivative
{
  "best_relative_error": 2.9770593621822814e-05,
  "tolerance": 0.005
}
[PASS] order 4 conductivity directional derivative
{
  "best_relative_error": 1.670821338090047e-05,
  "tolerance": 0.005
}
[PASS] order 4 CPML material-gradient exclusion
{
  "er_boundary_absmax": 0.0,
  "se_boundary_absmax": 0.0
}
[PASS] order 8 relative-permittivity directional derivative
{
  "best_relative_error": 5.614199239739206e-05,
  "tolerance": 0.005
}
[PASS] order 8 conductivity directional derivative
{
  "best_relative_error": 4.1479935392671076e-05,
  "tolerance": 0.005
}
[PASS] ord

In [4]:
vu.save_report(
    "03_gradient_2d",
    CHECKS,
    METADATA,
    extra={"directional_derivative_rows": all_rows},
)
print(f"Completed {len(CHECKS)} required checks.")


Report written to /Users/llsra/Desktop/DeepGPR/tests/results/03_gradient_2d.json
Completed 9 required checks.
